# Шаг 02. Проверка разметки 100 фотографий

В `annotations/worker_reference_first_pass.json` сохранён результат визуальной разметки 100 отдельных CC0-фотографий из `sampling_runs/20261108T001420_435829Z/worker_training_exam_draft.jsonl`. Для каждой фотографии записаны `photo_id`, ссылка на оригинал и рамки взрослых бабочек. `boxes: []` обозначает отсутствие размеченных взрослых бабочек.

Координаты `left`, `top`, `width` и `height` нормализованы относительно ширины и высоты оригинала. Рамки охватывают видимые крылья, тело, усики и ноги. Замечания к 10 отдельным случаям сохранены в `annotations/worker_reference_review_notes.json`.

Ноутбук сопоставляет разметку с исходным списком, проверяет уникальность ID, отсутствие пересечений с основной и конкурсной выборками, структуру записей и границы рамок.


In [ ]:
from pathlib import Path
import json
from collections import Counter

ROOT = Path.cwd()
RUN = "20261108T001420_435829Z"
MANIFEST = ROOT / "sampling_runs" / RUN / "worker_training_exam_draft.jsonl"
ANNOTATIONS = ROOT / "annotations" / "worker_reference_first_pass.json"
REVIEW_NOTES = ROOT / "annotations" / "worker_reference_review_notes.json"
CONTEST = ROOT / "dataset.jsonl"
MAIN = ROOT / "sampling_runs" / RUN / "main_dataset_draft.jsonl"

read_jsonl = lambda path: [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]
manifest = read_jsonl(MANIFEST)
annotations = json.loads(ANNOTATIONS.read_text(encoding="utf-8"))
review_notes = json.loads(REVIEW_NOTES.read_text(encoding="utf-8"))
contest = read_jsonl(CONTEST)
main = read_jsonl(MAIN)
print("Фотографий:", len(annotations), "Приоритетных для проверки:", len(review_notes["priority_review"]))

In [ ]:
import math

assert len(manifest) == len(annotations) == 100
manifest_ids = [row["photo_id"] for row in manifest]
annotation_ids = [row["photo_id"] for row in annotations]
assert len(set(manifest_ids)) == len(set(annotation_ids)) == 100
assert set(manifest_ids) == set(annotation_ids)
assert not set(annotation_ids) & {row["photo_id"] for row in contest}
assert not set(annotation_ids) & {row["photo_id"] for row in main}
assert not {row["photo_url"] for row in annotations} & {row["photo_url"] for row in contest}
assert not {row["photo_url"] for row in annotations} & {row["photo_url"] for row in main}
manifest_by_id = {row["photo_id"]: row for row in manifest}
for row in annotations:
    assert set(row) == {"photo_id", "photo_url", "boxes"}
    assert row["photo_url"] == manifest_by_id[row["photo_id"]]["photo_url"]
    assert isinstance(row["boxes"], list)
    for box in row["boxes"]:
        assert set(box) == {"left", "top", "width", "height"}
        assert all(isinstance(box[key], (int, float)) and math.isfinite(box[key]) for key in box)
        assert 0 <= box["left"] < 1 and 0 <= box["top"] < 1
        assert 0 < box["width"] <= 1 and 0 < box["height"] <= 1
        assert box["left"] + box["width"] <= 1
        assert box["top"] + box["height"] <= 1
print("Проверки пройдены. С бабочками:", sum(bool(row["boxes"]) for row in annotations),
      "без бабочек:", sum(not row["boxes"] for row in annotations),
      "рамок:", sum(len(row["boxes"]) for row in annotations))

## Визуальное представление разметки

Функция `show_examples(start=0, count=12)` отображает 12 фотографий, начиная с первой. Параметр `start` задаёт начальный индекс, `count` — число снимков. Красные рамки соответствуют координатам из `annotations/worker_reference_first_pass.json`; под каждым снимком указаны ID, число рамок и ссылка на оригинал.

Полная галерея текущей разметки — [worker_reference_annotated_gallery.html](worker_reference_annotated_gallery.html). Она содержит все 100 фотографий с красными рамками, по четыре фотографии в ряд.


In [ ]:
from html import escape
from IPython.display import HTML, display

def show_examples(start=0, count=12):
    if not (0 <= start < len(annotations)):
        raise ValueError("start вне диапазона")
    cards = []
    for row in annotations[start:start + count]:
        photo_id = row["photo_id"]
        url = escape(row["photo_url"], quote=True)
        overlays = "".join(
            f'<div style="position:absolute;left:{100*b["left"]}%;top:{100*b["top"]}%;'
            f'width:{100*b["width"]}%;height:{100*b["height"]}%;'
            'border:2px solid #e11;box-sizing:border-box;pointer-events:none"></div>'
            for b in row["boxes"]
        )
        cards.append(
            f'<div style="width:360px;min-width:360px">'
            f'<div style="position:relative;display:block"><img src="{url}" '
            'style="width:100%;display:block"/>' + overlays + '</div>'
            f'<div><b>{photo_id}</b> · рамок: {len(row["boxes"])} · '
            f'<a href="{url}" target="_blank">оригинал</a></div></div>'
        )
    display(HTML('<div style="display:flex;flex-wrap:wrap;gap:16px">' + ''.join(cards) + '</div>'))

show_examples(start=0, count=12)

## Результаты проверки

- В разметке 100 фотографий: 58 с бабочками и 42 с пустым списком рамок.
- Всего сохранена 61 рамка.
- ID и ссылки совпадают с исходным списком; повторяющихся ID и пересечений с основной и конкурсной выборками нет.
- Структура записей и нормализованные координаты прошли проверки; все рамки находятся в границах изображения.
